## GCS Intervention — proximity guard (collision avoidance)

A single drone flies an AUTO mission across the city, on a leg that runs straight
into the `radio_tower_intervention` standing on the corridor.

The GCS watches the drone's position and acts as a **guard**: while the drone is
within `radius` of the tower it **takes over** (GUIDED + a guided detour); once
the drone is clear past `release_radius` it **releases control**, and ArduPilot
resumes the paused mission from where it left off. If the drone comes back into
the zone, the GCS engages again.

The trigger is evaluated every monitor tick, so a `near` condition can go false
again — unlike `seq`/`after`/`dwell`, which are monotone and give the original
take-over-and-keep-it behaviour.

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, Color, Model
from simulator.entities import Intervention, ProximityTrigger, SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.helpers.processes import SimProcess
from simulator.planner import AutoPlan, InterventionPlan
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin and 

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, 0, 0, 0)

## Waypoints

In [ ]:
cruise_alt = 15.0
home_wp = ENU(x=0, y=0, z=0)
headup_wp = ENU(x=0, y=0, z=cruise_alt)
end_wp = ENU(x=-85, y=56, z=cruise_alt)
mission_wps = [home_wp, headup_wp, end_wp]

## Vehicle

In [ ]:
sysid = 1
model = Model.IRIS

mission_path = DATA_PATH / "missions" / "gcs_intervention.waypoints"

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=str(mission_path),
    navigation_speed=3.0,
    firmware=model.firmware,
)

vehicle = SimVehicle.from_relative(
    sysid=sysid,
    plan=plan,
    color=Color.BLUE,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    model=model,
)


## GCS

In [ ]:
gcs = SimGCS(name=f"{Color.BLUE.name}_{Color.BLUE.emoji}")
gcs.add_vehicle(vehicle)

### Intervention

`gcs.intervene` requires the GCS to already monitor the vehicle.

A **`ProximityTrigger`** is a *guard*: reversible, so the GCS hands control back
when the drone leaves the zone.

* The `radio_tower_intervention` in the world sits on the mission corridor: the straight leg (0,0) -> (-85,56) passes through (-47, 31), so the AUTO mission flies the drone straight into it.
* Guard geometry. `radius` is where the GCS takes over; `release_radius` is where it hands back (hysteresis, so it does not chatter at the boundary).
* Where the GCS steers the drone while it holds control: offset perpendicular to the corridor, far enough that once released the resumed mission leg no longer re-enters the guarded zone (otherwise AUTO would fly straight back in).

In [ ]:
obstacle = ENU(x=-47, y=30, z=cruise_alt)
guard_radius = 25.0  # meters
release_radius = 45.0  # meters
detour = ENU(x=-22, y=68, z=cruise_alt)

print(
    f"Obstacle at {obstacle.short()}, guard {guard_radius:.0f} m, release "
    f"{release_radius:.0f} m"
)

In [ ]:
gcs.intervene(
    vehicle,
    Intervention(
        trigger=ProximityTrigger(
            near=obstacle,
            radius=guard_radius,
            release_radius=release_radius,
        ),
        plan=InterventionPlan.from_relative_path(
            relative_path=[detour],
            enu_origin=enu_origin,
            relative_home=home,
            firmware=model.firmware,
            land=False,
        ),
        firmware=model.firmware,
    ),
)

## Oracle

In [ ]:
orac = Oracle()
orac.add_gcs(gcs)

## Visualizer

In [ ]:
gaz = Gazebo(
    gra_origin,
    world_path="simulator/visualizer/gazebo/worlds/small_city_intervention.world",
)
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
# The zone the GCS guards: inside the red sphere it takes over.
guard_marker = GazMarker(
    name="guard_zone",
    group="guard",
    pos=obstacle,
    color=Color.RED,
    radius=guard_radius,
    alpha=0.6,
)
# Where it steers the drone while it holds control.
detour_marker = GazMarker(
    name="detour",
    group="targets",
    pos=detour,
    color=Color.YELLOW,
)
for marker in [origin_marker, guard_marker, detour_marker]:
    gaz.markers.append(marker)

## Simulator

In [ ]:
simulator = Simulator(
    oracle=orac, visualizer=gaz, verbose=1, terminals=[SimProcess.LOGIC, SimProcess.GCS]
)

simulator.preview()


In [ ]:
simulator.run()


In [ ]:
orac.plot_trajectories(azim=-70, elev=15);
